In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("MinIO-PostgreSQL")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars", "/home/jovyan/postgresql-42.7.1.jar")  # PostgreSQL JDBC driver
    .getOrCreate()
)

hconf = spark.sparkContext._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", "http://minio:9000")
hconf.set("fs.s3a.access.key", "matrix")
hconf.set("fs.s3a.secret.key", "matrix123")
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "false")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

In [2]:
bronze_path = "s3a://bronze/"
silver_path = "s3a://silver/"
gold_path = "s3a://gold/"

In [3]:
import os
file_path = "/home/jovyan/work"
files = os.listdir(file_path)
print(files)


['docker-compose.yaml', 'spark-defaults.conf', 'notebooks', 'output.txt', 'Untitled(7)(3)(1).ipynb', 'spark1.ipynb', 'spark1.bash']


In [17]:
df_card = spark.read.csv(
    "/home/jovyan/work/card_trn.csv", 
    header=True, 
    inferSchema=True
)

df_customer = spark.read.csv(
    "/home/jovyan/work/cust.csv", 
    header=True, 
    inferSchema=True

)

In [18]:

df_card.write.mode("overwrite").csv(
    bronze_path + "card_trn.csv",
    header=True
)

df_customer.write.mode("overwrite").csv(
    bronze_path + "cust.csv",
    header=True
)

In [19]:
df_card.printSchema()
df_card.show(5, truncate=False)
df_customer.printSchema()
df_customer.show(5, truncate=False)

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- txn_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- merchant: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = true)

+--------------+-----------+----------+------+--------+-------+--------+-------+
|transaction_id|customer_id|txn_date  |amount|merchant|channel|currency|status |
+--------------+-----------+----------+------+--------+-------+--------+-------+
|2001          |1          |2024-10-01|150.25|Amazon  |ONLINE |USD     |SUCCESS|
|2002          |2          |2024-10-02|75.0  |Zara    |POS    |AZN     |SUCCESS|
|2003          |3          |2024-10-03|300.1 |Apple   |ONLINE |USD     |FAILED |
|2004          |4          |2024-10-04|50.75 |Bolt    |ONLINE |AZN     |SUCCESS|
|2005          |5          |2024-10-05|120.0 |Uber    |ONLINE |AZN     |SUCCESS|
+--------------+------

In [22]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, DoubleType, StringType, BooleanType

card_schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("txn_date", DateType(), True),
    StructField("amount", DoubleType(), True),
    StructField("merchant", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("status", StringType(), True)
])

customer_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", StringType(), True),  
    StructField("signup_date", DateType(), True),
    StructField("balance", StringType(), True),  
    StructField("vip_status", BooleanType(), True)
])

In [43]:
df_customer_bronze = spark.read.csv(
    bronze_path + "cust.csv", 
    schema=customer_schema, 
    header=True
)

df_card_bronze = spark.read.csv(
    bronze_path + "card_trn.csv",
    schema=card_schema,  
    header=True
)

In [44]:
df_step1 = df_customer_bronze.withColumn(
    "balance",
    F.when(F.col("balance").cast("double").isNull(), 0.0)
     .otherwise(F.col("balance").cast("double"))
)

In [45]:
df_step1.select("customer_id", "name", "balance").show(10, truncate=False)

+-----------+------+-------+
|customer_id|name  |balance|
+-----------+------+-------+
|1          |Nazim |1000.5 |
|2          |Samir |200.0  |
|3          |Shahin|350.75 |
|4          |Aysel |1500.0 |
|5          |Rauf  |0.0    |
|6          |Murad |950.0  |
|7          |Leyla |1200.0 |
|8          |Orxan |800.1  |
|9          |Fidan |0.0    |
|10         |Tural |700.0  |
+-----------+------+-------+



In [46]:
df_step2 = df_step1.withColumn(
    "age_int", 
    F.col("age").cast("int") 
).withColumn(
    "age_group",
    F.when(F.col("age_int") < 25, "young")
     .when((F.col("age_int") >= 25) & (F.col("age_int") <= 40), "adult")
     .when(F.col("age_int") > 40, "senior")
     .otherwise("unknown")  
)

In [47]:
df_step2.select("customer_id", "name", "age", "age_int", "age_group").show(10, truncate=False)

+-----------+------+----+-------+---------+
|customer_id|name  |age |age_int|age_group|
+-----------+------+----+-------+---------+
|1          |Nazim |25  |25     |adult    |
|2          |Samir |NULL|NULL   |unknown  |
|3          |Shahin|30  |30     |adult    |
|4          |Aysel |xyz |NULL   |unknown  |
|5          |Rauf  |28  |28     |adult    |
|6          |Murad |32  |32     |adult    |
|7          |Leyla |22  |22     |young    |
|8          |Orxan |45  |45     |senior   |
|9          |Fidan |29  |29     |adult    |
|10         |Tural |40  |40     |adult    |
+-----------+------+----+-------+---------+



In [48]:
df_step3 = df_step2.withColumnRenamed("vip_status", "flg_is_vip")

In [49]:
df_step3.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- balance: double (nullable = true)
 |-- flg_is_vip: boolean (nullable = true)
 |-- age_int: integer (nullable = true)
 |-- age_group: string (nullable = false)



In [50]:
df_step4 = df_step3.withColumn(
    "insert_date", 
    F.current_date()  )

In [51]:
df_step4.select("customer_id", "name", "insert_date").show(5, truncate=False)

+-----------+------+-----------+
|customer_id|name  |insert_date|
+-----------+------+-----------+
|1          |Nazim |2026-02-17 |
|2          |Samir |2026-02-17 |
|3          |Shahin|2026-02-17 |
|4          |Aysel |2026-02-17 |
|5          |Rauf  |2026-02-17 |
+-----------+------+-----------+
only showing top 5 rows



In [52]:
df_customer_silver = df_step4.drop("age_int")

In [53]:
silver_path = "s3a://silver/"
df_customer_silver.write.mode("overwrite").option("overwriteSchema", "true").parquet(
    silver_path + "customer"
)

In [54]:
df_customer_silver.printSchema()
df_customer_silver.show(10, truncate=False)

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- balance: double (nullable = true)
 |-- flg_is_vip: boolean (nullable = true)
 |-- age_group: string (nullable = false)
 |-- insert_date: date (nullable = false)

+-----------+------+----+-----------+-------+----------+---------+-----------+
|customer_id|name  |age |signup_date|balance|flg_is_vip|age_group|insert_date|
+-----------+------+----+-----------+-------+----------+---------+-----------+
|1          |Nazim |25  |2024-01-10 |1000.5 |true      |adult    |2026-02-17 |
|2          |Samir |NULL|2024-02-15 |200.0  |false     |unknown  |2026-02-17 |
|3          |Shahin|30  |2024-03-20 |350.75 |true      |adult    |2026-02-17 |
|4          |Aysel |xyz |2024-04-05 |1500.0 |true      |unknown  |2026-02-17 |
|5          |Rauf  |28  |2024-05-01 |0.0    |false     |adult    |2026-02-17 |
|6          |Murad |32  |2024-06-10 |950

In [55]:
df_card_silver = df_card_bronze.withColumn(
    "insert_date",
    F.current_date()
)

In [56]:
df_card_silver.write.mode("overwrite").option("overwriteSchema", "true").parquet(
    silver_path + "card_transactions"
)

In [57]:
df_card_silver.show(5, truncate=False)

+--------------+-----------+----------+------+--------+-------+--------+-------+-----------+
|transaction_id|customer_id|txn_date  |amount|merchant|channel|currency|status |insert_date|
+--------------+-----------+----------+------+--------+-------+--------+-------+-----------+
|2001          |1          |2024-10-01|150.25|Amazon  |ONLINE |USD     |SUCCESS|2026-02-17 |
|2002          |2          |2024-10-02|75.0  |Zara    |POS    |AZN     |SUCCESS|2026-02-17 |
|2003          |3          |2024-10-03|300.1 |Apple   |ONLINE |USD     |FAILED |2026-02-17 |
|2004          |4          |2024-10-04|50.75 |Bolt    |ONLINE |AZN     |SUCCESS|2026-02-17 |
|2005          |5          |2024-10-05|120.0 |Uber    |ONLINE |AZN     |SUCCESS|2026-02-17 |
+--------------+-----------+----------+------+--------+-------+--------+-------+-----------+
only showing top 5 rows



In [58]:
df_customer_silver = spark.read.parquet(silver_path + "customer")
df_card_silver = spark.read.parquet(silver_path + "card_transactions")

In [59]:
df_total_amount = df_card_silver.groupBy("customer_id") \
    .agg(F.sum("amount").alias("total_amount"))

In [60]:
df_total_amount.show(10, truncate=False)

+-----------+------------+
|customer_id|total_amount|
+-----------+------------+
|1          |270.25      |
|6          |315.4       |
|3          |800.85      |
|5          |220.0       |
|9          |135.25      |
|4          |250.75      |
|8          |580.0       |
|7          |135.0       |
|10         |340.5       |
|2          |105.0       |
+-----------+------------+



In [61]:
df_result = df_total_amount.join(
    df_customer_silver.select("customer_id", "name"),
    on="customer_id",
    how="inner"
)

In [62]:
df_result.select("customer_id", "name", "total_amount").show(10, truncate=False)

+-----------+------+------------+
|customer_id|name  |total_amount|
+-----------+------+------------+
|1          |Nazim |270.25      |
|2          |Samir |105.0       |
|3          |Shahin|800.85      |
|4          |Aysel |250.75      |
|5          |Rauf  |220.0       |
|6          |Murad |315.4       |
|7          |Leyla |135.0       |
|8          |Orxan |580.0       |
|9          |Fidan |135.25      |
|10         |Tural |340.5       |
+-----------+------+------------+



In [63]:
df_top2 = df_result.select("name", "total_amount") \
    .orderBy(F.col("total_amount").desc()) \
    .limit(2)

In [64]:
df_top2.show(truncate=False)

+------+------------+
|name  |total_amount|
+------+------------+
|Shahin|800.85      |
|Orxan |580.0       |
+------+------------+



In [65]:
df_top2.write.mode("overwrite").parquet(gold_path + "top_customers")